# L2 detection and ingestion quality metrics

Run all cells in Google Colab. Input files are harmonized `processed/<sensor>_<year>.parquet` files on Drive. Outputs are the ten `l2_*` tables and `_log_lote2.csv`.

Use `GITHUB_TOKEN` in Colab Secrets if the repository is private. Set `YEARS` explicitly to require a complete year selection. With `YEARS = None`, the notebook processes available files and ADD-04 reports gaps within each instrument's observed year range. Missing files and countries without detections cannot be distinguished.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64
import os
import subprocess
import sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Sprinklers')
REPO = Path('/content/repo')
env = os.environ.copy()
try:
    token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    token = None
if token:
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    env.update(GIT_CONFIG_COUNT='1',
               GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
               GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {credential}')
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], env=env, check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git', str(REPO)], env=env, check=True)
del env, token
sys.path.insert(0, str(REPO / 'backend'))


In [ ]:
import json
import importlib
import re
from datetime import datetime, timezone
import pandas as pd
from metrics import detections, ingestion_qa

importlib.reload(detections)
importlib.reload(ingestion_qa)

SENSORS = ['modis', 'viirs-jpss1']
YEARS = None  # Example: [2018, 2019]. None selects available files.
FORCE = False
CONFIDENCE = ('n', 'h')
TYPE_VALUE = 0
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
METRICS.mkdir(parents=True, exist_ok=True)
PARTS = METRICS / '_l2_parts'
COLUMNS = ['country', 'instrument', 'satellite', 'version', 'latitude', 'longitude',
           'acq_date', 'acq_time', 'acq_datetime_utc', 'daynight', 'type',
           'confidence_raw', 'confidence_class', 'pixel_area_km2', 'frp']
FUNCTIONS = {
    'l2_obs04_night_pct': detections.obs04_night_pct,
    'l2_obs05_frp_median': detections.obs05_frp_median,
    'l2_obs06_frp_p90': detections.obs06_frp_p90,
    'l2_obs07_frp_peak': detections.obs07_frp_peak,
    'l2_obs08_frp_availability': detections.obs08_frp_availability,
    'l2_obs09_type_distribution': detections.obs09_type_distribution,
    'l2_obs10_pixel_area_median': detections.obs10_pixel_area_median,
    'l2_qa01_schema_validation': ingestion_qa.qa01_schema_validation,
    'l2_add03_duplicates': ingestion_qa.add03_duplicates,
}
FILTERED = set(list(FUNCTIONS)[:5])
PARAMETERS = {'confidence': list(CONFIDENCE), 'type_value': TYPE_VALUE, 'grid_deg': None}
CODE_REVISION = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()

def atomic_parquet(frame, path):
    temporary = path.with_suffix('.parquet.tmp')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)

inputs = []
for path in sorted(PROCESSED.glob('*.parquet')):
    match = re.fullmatch(r'(.+)_(\d{4})', path.stem)
    if match and match[1] in SENSORS:
        year = int(match[2])
        if YEARS is None or year in YEARS:
            inputs.append((match[1], year, path))
if not inputs:
    raise FileNotFoundError(f'No selected processed Parquet files in {PROCESSED}')
if YEARS is not None:
    available = {(sensor, year) for sensor, year, _ in inputs}
    missing = [(sensor, year) for sensor in SENSORS for year in YEARS if (sensor, year) not in available]
    if missing:
        raise FileNotFoundError(f'Missing requested sensor/year files: {missing}')


In [ ]:
part_directories = []
for sensor, year, path in inputs:
    directory = PARTS / f'{sensor}_{year}'
    directory.mkdir(parents=True, exist_ok=True)
    marker = directory / '_SUCCESS.json'
    signature = {'source_size': path.stat().st_size, 'source_mtime_ns': path.stat().st_mtime_ns,
                 'parameters': PARAMETERS, 'revision': CODE_REVISION}
    expected = [directory / f'{name}.parquet' for name in FUNCTIONS]
    complete = marker.exists() and all(part.exists() for part in expected)
    cached_signature = None
    if complete:
        try:
            cached_signature = json.loads(marker.read_text())
        except (OSError, ValueError):
            pass  # A damaged completion marker requires recalculation.
    if not FORCE and complete and cached_signature == signature:
        part_directories.append(directory)
        print(f'Reusing completed {sensor} {year}')
        continue
    marker.unlink(missing_ok=True)
    frame = pd.read_parquet(path, columns=COLUMNS)
    for column in ['country', 'instrument', 'satellite', 'version', 'daynight', 'confidence_class']:
        frame[column] = frame[column].astype('category')
    for name, function in FUNCTIONS.items():
        options = dict(confidence=CONFIDENCE, type_value=TYPE_VALUE) if name in FILTERED else {}
        result = function(frame, **options)
        if name == 'l2_qa01_schema_validation':
            result.insert(1, 'year', year)
        atomic_parquet(result, directory / f'{name}.parquet')
    temporary_marker = marker.with_suffix('.json.tmp')
    temporary_marker.write_text(json.dumps(signature, sort_keys=True))
    temporary_marker.replace(marker)
    part_directories.append(directory)
    del frame
    print(f'Processed {sensor} {year}')


In [ ]:
tables = {}
log = []
run_time = datetime.now(timezone.utc).isoformat()
for name in FUNCTIONS:
    result = pd.concat([pd.read_parquet(directory / f'{name}.parquet')
                        for directory in part_directories], ignore_index=True)
    keys = [column for column in ['country', 'year', 'instrument', 'satellite', 'version', 'type_label']
            if column in result.columns]
    result = result.sort_values(keys).reset_index(drop=True)
    atomic_parquet(result, METRICS / f'{name}.parquet')
    tables[name] = result

counts = tables['l2_add03_duplicates'].groupby(
    ['country', 'instrument', 'year'], observed=True, dropna=False
).n_rows.sum().reset_index()
name = 'l2_add04_zero_detection_countries'
tables[name] = ingestion_qa.add04_zero_detection_countries(counts)
atomic_parquet(tables[name], METRICS / f'{name}.parquet')
for name, result in tables.items():
    log.append({'metric': name, 'parameters': json.dumps(PARAMETERS, sort_keys=True),
                'sensor_years': json.dumps([(sensor, year) for sensor, year, _ in inputs]),
                'output_rows': len(result), 'executed_at_utc': run_time, 'revision': CODE_REVISION})
log_path = METRICS / '_log_lote2.csv'
temporary = log_path.with_suffix('.csv.tmp')
pd.DataFrame(log).to_csv(temporary, index=False)
temporary.replace(log_path)


In [ ]:
qa = tables['l2_qa01_schema_validation']
assert (qa.n_valid + qa.n_invalid).equals(qa.n_rows)
availability = tables['l2_obs08_frp_availability']
assert availability[['n_frp_usable', 'n_frp_null', 'n_frp_zero', 'n_frp_negative']].sum(axis=1).equals(availability.n_eligible)
types = tables['l2_obs09_type_distribution']
totals = types.groupby(['country', 'year', 'instrument', 'satellite', 'version'], observed=True, dropna=False).pct.sum()
assert totals.sub(100).abs().lt(1e-9).all()
for name, result in tables.items():
    print(name, result.shape)
    print(result.dtypes)
    print(result.head(3).to_string(index=False))
print(f'L2 completed: {len(tables)} tables in {METRICS}')
